# Exact NLP benchmark (global) via SCIP

Solves the aggregator bidding problem with the **true analytical flexibility
purchase cost** (Eq. 10c) instead of a ReLU-NN surrogate, to global optimality
using SCIP (spatial branch-and-bound). Produces a reference optimum against
which the surrogate methods (Gurobi ML, PCAR, PCTAR, FICNN, PWL) are compared.

**Consistency with the surrogate runs**
- Same inputs, paths, dataclasses and result-saving convention as the Gurobi notebook.
- Rebound coupling identical: `xtil_t = xbar_t - sum_j A[j][t]*x_j*activation[j]`.
- Singularity handled with a **multiplicative** margin `x_t <= 0.98 * xtil_t`
  (matches the NN training domain), replacing the additive epsilon.
- True cost uses the zero-floor: `cost_t = max(0, (x_t/r_t)(q_t - ln(xtil_t/x_t - 1)))`,
  encoded without a binary by relying on cost being minimised in the objective.
- q = shaping_parameters[0], r = shaping_parameters[1], used **raw / unscaled**.


In [1]:
# Imports for pythonifying code

from typing import Callable, List, Dict, Any

# Path related imports

import os
import sys

In [2]:
# Append utilities path

current_path = os.getcwd()
cwd = os.path.dirname(current_path)

sys.path.append(os.path.join(cwd, "utilities"))

## Get input data

In [3]:
# Import pickle operations

from pickle_file_operations import read_pickle, write_pickle

# Import dataclasses

from optimization_dataclasses import Prosumer_Cluster_Data, Market_Data, Bivariate_Pwl_Input, Layer_Data, Optimization_Results
from dataclasses import fields

In [4]:
# Numerics / dataframes

import numpy as np
import pandas as pd
import time as Time

## SCIP imports

In [5]:
# PySCIPOpt global NLP solver
# Install (once):  pip install pyscipopt   (or: conda install -c conda-forge pyscipopt)

from pyscipopt import Model, quicksum, log

## Model configuration

In [6]:
# ---- Config (mirrors the Gurobi notebook) ----------------------------------

MARGIN = 0.98        # multiplicative singularity margin:  x_t <= MARGIN * xtil_t
                     # (matches NN training domain x <= 0.98 * xtil)

X_LB = 0.098         # lower bound on flexibility_MWh          (from Gurobi notebook)
XTIL_LB = 0.10       # lower bound on updated_max_flexibility  (from Gurobi notebook)

TIME_LIMIT = 3600.0  # match the surrogate runs' 3600 s budget.
                     # For the "double the budget" robustness framing on a subset, use 7200.0.
GAP_LIMIT = 0.01      # 0.0 => solve to global optimality (or hit the time limit).
                     # NOTE: surrogate runs used MIPGap=0.01; here we ask for the
                     # true global optimum, so we set 0.0 and report the achieved gap.

## Market variables and constraints (SCIP)

In [7]:
def _add_market_variables(opt_model: Model, time_steps: int):
    """Continuous decision variables, matching the Gurobi notebook's names/bounds."""
    variable_names = ["flexibility_MWh", "updated_max_flexibility_MWh", "flexibility_cost_DKK"]
    lower_bounds = [X_LB, XTIL_LB, 0.0]

    decision_variables = {}
    for time in range(time_steps):
        decision_variables[time] = {
            "flexibility_MWh": opt_model.addVar(
                vtype="C", lb=lower_bounds[0], ub=None, name=f"flexibility_MWh_{time}"),
            "updated_max_flexibility_MWh": opt_model.addVar(
                vtype="C", lb=lower_bounds[1], ub=None, name=f"updated_max_flexibility_MWh_{time}"),
            "flexibility_cost_DKK": opt_model.addVar(
                vtype="C", lb=lower_bounds[2], ub=None, name=f"flexibility_cost_DKK_{time}"),
        }
    return decision_variables


def _add_market_constraints(opt_model: Model, decision_variables: Dict[int, Any],
                            prosumer_data: Prosumer_Cluster_Data,
                            activation_signal: np.ndarray, margin: float = MARGIN):
    """Rebound coupling + multiplicative singularity margin.

    Rebound (identical indexing to the Gurobi notebook):
        xtil_t = xbar_t - sum_j A[j][t] * x_j * activation[j]
    Margin (multiplicative, replaces additive epsilon):
        x_t <= margin * xtil_t
    """
    cross_temporal_matrix = prosumer_data.cross_temporal_matrix
    maximum_flexibility_profile = prosumer_data.max_flexibility_profile
    time_steps = len(maximum_flexibility_profile)

    for time in range(time_steps):
        variables = decision_variables[time]

        # Multiplicative margin
        opt_model.addCons(
            variables["flexibility_MWh"] <= margin * variables["updated_max_flexibility_MWh"],
            name=f"margin_{time}",
        )

        # Rebound / cross-temporal availability
        opt_model.addCons(
            variables["updated_max_flexibility_MWh"]
            == maximum_flexibility_profile[time]
            - quicksum(
                cross_temporal_matrix[j][time]
                * decision_variables[j]["flexibility_MWh"]
                * activation_signal[j]
                for j in range(time_steps)
            ),
            name=f"rebound_{time}",
        )
    return None

## True analytical cost (Eq. 10c) with zero floor

`cost_t = max(0, (x_t / r_t) * (q_t - ln(xtil_t / x_t - 1)))`

Encoded as two lower bounds. Because `cost_t` is **minimised** in the objective
(it enters profit with a minus sign and appears nowhere else), the optimum drives
`cost_t` down to the tighter of `0` and the analytical expression — i.e. exactly
`max(0, expr)` — with no binary variable required.

In [8]:
def _add_true_cost_constraints(opt_model: Model, decision_variables: Dict[int, Any],
                               pwl_data: Dict[int, Bivariate_Pwl_Input], time_steps: int):
    """Impose cost_t >= max(0, analytical). Zero floor via lb=0 already set."""
    for time in range(time_steps):
        variables = decision_variables[time]
        q = pwl_data[time].shaping_parameters[0]   # q_t (raw, unscaled)
        r = pwl_data[time].shaping_parameters[1]   # r_t (raw, unscaled)

        x = variables["flexibility_MWh"]
        xtil = variables["updated_max_flexibility_MWh"]

        # cost_t >= (x/r) * (q - ln(xtil/x - 1))
        opt_model.addCons(
            variables["flexibility_cost_DKK"] >= (x / r) * (q - log(xtil / x - 1.0)),
            name=f"cost_def_{time}",
        )
    return None

## Objective

In [9]:
def true_cost_objective(opt_model: Model, decision_variables: Dict[int, Any],
                        market_data: Market_Data, time_steps: int):
    """max sum_t ( price_t * x_t - cost_t )."""
    opt_model.setObjective(
        quicksum(
            market_data.market_price[time] * decision_variables[time]["flexibility_MWh"]
            - decision_variables[time]["flexibility_cost_DKK"]
            for time in range(time_steps)
        ),
        sense="maximize",
    )
    return None

## Results helpers

In [10]:
def get_results(pwl_data, market_data, market_variables, time_steps):
    """Same schema as the Gurobi notebook so downstream analysis is unchanged."""
    variable_names = ["flexibility_MWh", "updated_max_flexibility_MWh", "flexibility_cost_DKK"]

    results_dict = {
        variable_name: np.array(
            [opt_model_val(market_variables[time][variable_name]) for time in range(time_steps)]
        )
        for variable_name in variable_names
    }
    results_dict["market_price"] = market_data.market_price
    results_dict["shaping_parameters"] = [pwl_data[time].shaping_parameters for time in range(time_steps)]
    return pd.DataFrame(results_dict)


def true_cost_vector(x, xtil, q, r):
    """Analytical cost with zero floor, for independent verification."""
    x = np.asarray(x, dtype=float); xtil = np.asarray(xtil, dtype=float)
    q = np.asarray(q, dtype=float); r = np.asarray(r, dtype=float)
    raw = (x / r) * (q - np.log(xtil / x - 1.0))
    return np.maximum(0.0, raw)

In [11]:
# SCIP value accessor used inside get_results (needs the solved model in scope).
# We bind it per-solve via a small closure to keep get_results signature identical.

_CURRENT_MODEL = {"m": None}

def opt_model_val(var):
    return _CURRENT_MODEL["m"].getVal(var)

## Output folder

Saved alongside the surrogate results, in a new `Exact NLP` folder, using the
same `Optimization_Results` dataclass and `Results.pickle` filename.

In [12]:
def makedir(path):
    if not os.path.exists(path):
        os.mkdir(path)

results_path = os.path.join(cwd, "price scenario results")
makedir(results_path)

nlp_folder = "Exact NLP"
makedir(os.path.join(results_path, nlp_folder))

## Load inputs (same files as the Gurobi notebook)

In [13]:
# Get the inputs for the optimization problem

input_path = os.path.join(cwd, "optimization input")
market_data_dict = read_pickle(os.path.join(input_path, "price_scenarios.pickle"))
pwl_data = read_pickle(os.path.join(input_path, "5_pwl_data_input.pickle"))
prosumer_cluster_data = read_pickle(os.path.join(input_path, "prosumer_cluster_data.pickle"))
# nn_scaler NOT needed: the exact NLP evaluates the true function in raw units.

time_steps = 24
season = "Winter"

## Quick input sanity check (recommended before the full run)

Confirms the rebound orientation and margin feasibility on one scenario:
`xtil` should stay positive and below `xbar`, and there should be room for
`x >= 0.098` under `x <= 0.98 * xtil`.

In [14]:
# Peek at one scenario to validate A-orientation and bounds
_scn = list(market_data_dict.keys())[0]
_md = market_data_dict[_scn][0]
_pd = prosumer_cluster_data[season]

print("scenario:", _scn)
print("xbar (max_flexibility_profile):", np.round(_pd.max_flexibility_profile, 3))
print("A shape:", np.shape(_pd.cross_temporal_matrix),
      " column-sum max:", np.max(np.sum(_pd.cross_temporal_matrix, axis=0)))
print("activation_signal:", np.round(_md.activation_signal, 3))
print("q (shaping[0]) sample:", [round(pwl_data[t].shaping_parameters[0], 3) for t in range(3)])
print("r (shaping[1]) sample:", [round(pwl_data[t].shaping_parameters[1], 3) for t in range(3)])
# If x <= 0.98*xtil and xtil ~ xbar, need 0.98*xbar >= 0.098  -> xbar >= 0.10 (holds).

scenario: High_prices
xbar (max_flexibility_profile): [13.327 13.513 13.327 12.783 11.926  9.444  7.196  5.886  4.63   3.529
  2.671  2.127  1.987  2.127  2.671  3.529  4.63   5.886  7.196  9.444
 11.926 12.783 13.327 13.513]
A shape: (24, 24)  column-sum max: 2.666
activation_signal: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
q (shaping[0]) sample: [7, 4, 7]
r (shaping[1]) sample: [0.288, 0.714, 1.17]


## Single-scenario probe

Run ONE low-price scenario first to see whether SCIP closes the gap within the
time limit before committing to the full 30-case sweep. Report status, gap,
solve time, and the cost-encoding mismatch (should be ~1e-6).

In [15]:
def solve_one_case(market_data, prosumer_data, pwl_data, time_steps,
                   time_limit=TIME_LIMIT, gap_limit=GAP_LIMIT, verbose=True):
    opt_model = Model("exact_nlp_aggregator")
    _CURRENT_MODEL["m"] = opt_model

    market_variables = _add_market_variables(opt_model, time_steps)
    _add_market_constraints(opt_model, market_variables, prosumer_data,
                            market_data.activation_signal, MARGIN)
    _add_true_cost_constraints(opt_model, market_variables, pwl_data, time_steps)
    true_cost_objective(opt_model, market_variables, market_data, time_steps)

    opt_model.setParam("limits/time", time_limit)
    opt_model.setParam("limits/gap", gap_limit)
    if not verbose:
        opt_model.hideOutput()
    else:
        opt_model.setParam("display/verblevel", 4)

    start_cpu = Time.process_time()
    opt_model.optimize()
    end_cpu = Time.process_time()

    status = opt_model.getStatus()
    has_sol = opt_model.getNSols() > 0

    df = get_results(pwl_data, market_data, market_variables, time_steps) if has_sol else None

    # Independent verification of the objective with the true cost
    if has_sol:
        x = df["flexibility_MWh"].to_numpy()
        xtil = df["updated_max_flexibility_MWh"].to_numpy()
        q = np.array([pwl_data[t].shaping_parameters[0] for t in range(time_steps)])
        r = np.array([pwl_data[t].shaping_parameters[1] for t in range(time_steps)])
        tc = true_cost_vector(x, xtil, q, r)
        realised_profit = float(np.sum(market_data.market_price * x - tc))
        cost_mismatch = float(np.max(np.abs(df["flexibility_cost_DKK"].to_numpy() - tc)))
    else:
        realised_profit, cost_mismatch = float("nan"), float("nan")

    info = {
        "status": status,
        "objective": opt_model.getObjVal() if has_sol else float("nan"),
        "dual_bound": opt_model.getDualbound(),
        "gap": opt_model.getGap(),
        "solve_time": opt_model.getSolvingTime(),
        "cpu_time": end_cpu - start_cpu,
        "realised_profit": realised_profit,
        "cost_mismatch": cost_mismatch,
        "dataframe": df,
    }
    return info


# --- probe one scenario ---
# _probe_scn = list(market_data_dict.keys())[2]
# _probe = solve_one_case(market_data_dict[_probe_scn][0], prosumer_cluster_data[season],
#                         pwl_data, time_steps, time_limit=TIME_LIMIT, gap_limit=GAP_LIMIT,
#                         verbose=True)
# print("\n--- PROBE RESULT ---")
# for k in ["status", "objective", "dual_bound", "gap", "solve_time", "cpu_time",
#           "realised_profit", "cost_mismatch"]:
#     print(f"{k:15s}: {_probe[k]}")

## Full sweep over all scenarios and cases

Mirrors the Gurobi notebook's loop and save pattern. Wraps the SCIP results into
the same `Optimization_Results` dataclass. `mip_gap` stores the achieved global
optimality gap; a timed-out case will show a nonzero gap and status
`timelimit` — that itself is a reportable intractability result.

**Tip:** the full 30-case sweep at 3600 s each could be up to ~30 h of compute if
every case times out. Consider running a representative subset first (e.g. the
low-price scenario set), or lowering `TIME_LIMIT` for an initial pass.

In [16]:
%%time
results_dict = {}

for scenario, market_data_list in market_data_dict.items():
    results_dict[scenario] = {}
    for case, market_data in list(enumerate(market_data_list)):
        print(f"[NLP] scenario={scenario} case={case} ...", flush=True)

        info = solve_one_case(
            market_data, prosumer_cluster_data[season], pwl_data, time_steps,
            time_limit=TIME_LIMIT, gap_limit=GAP_LIMIT, verbose=False,
        )

        # Store the achieved global gap in mip_gap; status/realised profit kept in df attrs
        df = info["dataframe"]
        if df is not None:
            df.attrs["status"] = info["status"]
            df.attrs["objective"] = info["objective"]
            df.attrs["dual_bound"] = info["dual_bound"]
            df.attrs["realised_profit"] = info["realised_profit"]
            df.attrs["cost_mismatch"] = info["cost_mismatch"]

        optimization_results = Optimization_Results(
            decision_variables=df,
            runtime=info["solve_time"],
            cpu_time=info["cpu_time"],
            mip_gap=info["gap"],
        )

        results_dict[scenario][str(case)] = optimization_results

        print(f"      status={info['status']}  obj={info['objective']:.2f}  "
              f"gap={info['gap']*100:.3f}%  time={info['solve_time']:.1f}s  "
              f"realised={info['realised_profit']:.2f}  mismatch={info['cost_mismatch']:.2e}",
              flush=True)

        write_pickle(results_dict, os.path.join(results_path, nlp_folder, "Results.pickle"))

[NLP] scenario=High_prices case=0 ...
      status=gaplimit  obj=14121.69  gap=0.999%  time=1.0s  realised=14121.69  mismatch=9.00e-07
[NLP] scenario=High_prices case=1 ...
      status=gaplimit  obj=41307.30  gap=0.940%  time=0.0s  realised=41307.30  mismatch=8.99e-07
[NLP] scenario=High_prices case=2 ...
      status=gaplimit  obj=19960.53  gap=0.990%  time=0.0s  realised=19960.53  mismatch=9.00e-07
[NLP] scenario=High_prices case=3 ...
      status=gaplimit  obj=58605.48  gap=0.654%  time=0.0s  realised=58605.48  mismatch=9.00e-07
[NLP] scenario=High_prices case=4 ...
      status=gaplimit  obj=56777.24  gap=0.905%  time=0.0s  realised=56777.24  mismatch=9.00e-07
[NLP] scenario=High_prices case=5 ...
      status=gaplimit  obj=66586.72  gap=0.932%  time=0.0s  realised=66586.72  mismatch=9.00e-07
[NLP] scenario=High_prices case=6 ...
      status=gaplimit  obj=59456.99  gap=0.750%  time=0.0s  realised=59456.99  mismatch=9.00e-07
[NLP] scenario=High_prices case=7 ...
      status=gapl

## Notes

- **If SCIP raises a log-domain / evaluation error:** it means `xtil/x - 1` went
  non-positive somewhere during branch-and-bound. The margin `x <= 0.98*xtil`
  keeps `xtil/x - 1 >= 1/0.98 - 1 ≈ 0.0204 > 0` at feasible points, but if the
  error persists you can tighten the ratio bound explicitly, e.g. add
  `opt_model.addCons(xtil >= (1.0/MARGIN) * x)` (equivalent) or introduce an
  auxiliary `w_t = xtil_t/x_t - 1` with `w_t >= 0.0204` and `cost >= (x/r)*(q - log(w))`.
- **Reference optimum vs surrogates:** compare `df.attrs['realised_profit']`
  here against each surrogate's realised profit (recomputed with the true cost)
  to obtain the optimality gap / regret for Table 4.
- **Global proof vs intractability:** `status == 'optimal'` with `gap ~ 0` gives a
  certified optimum; `status == 'timelimit'` with a large gap is the evidence that
  the exact problem is intractable, justifying surrogate use (answers R1#6 either way).
